# MetaVSVCap: Pointer-Generator + Meta-Learning (Demo with Dummy Data)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

## Step 1: Simulate Dummy Graph Data and Captions

In [ ]:
def generate_dummy_graph(num_nodes=5, feature_dim=256):
    return torch.randn(num_nodes, feature_dim)

visual_graph = generate_dummy_graph()
semantic_graph = generate_dummy_graph()
rare_vocab = {'ukulele', 'slicing', 'zebra'}
tokenized_input = torch.tensor([1, 2, 3, 4])
copy_candidates = [5, 6, 7]

## Step 2: Graph Encoder and Cross-Graph Attention

In [ ]:
class DummyGraphEncoder(nn.Module):
    def forward(self, graph):
        return graph

def attention(query, context):
    scores = torch.matmul(context, query.unsqueeze(-1)).squeeze(-1)
    weights = F.softmax(scores, dim=0)
    return torch.sum(weights.unsqueeze(-1) * context, dim=0)

def combine(v, s):
    return (v + s) / 2

graph_encoder = DummyGraphEncoder()
v_emb = graph_encoder(visual_graph)
s_emb = graph_encoder(semantic_graph)
context = torch.stack([combine(v, attention(v, s_emb)) for v in v_emb])

## Step 3: Pointer-Generator Decoder

In [ ]:
class PointerGeneratorDecoder(nn.Module):
    def __init__(self, vocab_size=10, hidden_size=512, graph_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTMCell(hidden_size, hidden_size)
        self.graph_attn = nn.Linear(graph_dim, hidden_size)
        self.output_proj = nn.Linear(hidden_size, vocab_size)
        self.pointer_gate = nn.Linear(hidden_size + graph_dim, 1)

    def forward(self, tokens, context, copy_candidates):
        h_t = torch.zeros(1, 512)
        c_t = torch.zeros(1, 512)
        outputs = []
        for token in tokens:
            emb = self.embedding(token.unsqueeze(0))
            h_t, c_t = self.lstm(emb.squeeze(0), (h_t, c_t))
            context_vec = torch.mean(context, dim=0)
            attn_proj = self.graph_attn(context_vec)
            p_gen = F.softmax(self.output_proj(h_t), dim=-1)
            p_copy = F.softmax(torch.matmul(context, h_t.squeeze(0)), dim=0)
            lambda_ = torch.sigmoid(self.pointer_gate(torch.cat([h_t, context_vec.unsqueeze(0)], dim=-1)))
            final_dist = (1 - lambda_) * p_gen
            for i, idx in enumerate(copy_candidates):
                final_dist[0, idx] += lambda_ * p_copy[i]
            outputs.append(final_dist)
        return outputs

decoder = PointerGeneratorDecoder()
outputs = decoder(tokenized_input, context, copy_candidates)
for i, dist in enumerate(outputs):
    print(f"Step {i+1} prediction: ", torch.argmax(dist, dim=-1).item())